# FedMed-BN: Notebook 4 - Results Visualization & Thesis Figures
**Generate all plots for thesis Chapter 5**

In [ ]:
# Cell 1: Load all results (auto-detects SMALL vs FULL model runs)
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json
import os
import glob
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

results_dir = '/content/FedMed-BN/results'
models_dir = '/content/FedMed-BN/models'

# --- Auto-detect which run these results belong to ---
# If the "_small" files exist, we are looking at the BanglaBERT-small (Colab) run;
# otherwise we assume the full BanglaBERT (lab PC) run.
if os.path.exists(f'{results_dir}/centralized_results_small.json'):
    suffix = '_small'
    run_name = 'BanglaBERT-small'
else:
    suffix = ''
    run_name = 'BanglaBERT (full)'

print(f"Detected results for: {run_name}")
print(f"Using file suffix: '{suffix}'\n")

# --- Load centralized results ---
with open(f'{results_dir}/centralized_results{suffix}.json', 'r') as f:
    cent_results = json.load(f)
print(f"Centralized best F1: {cent_results['centralized']['best_f1']:.4f}")

# --- Load federated (no DP) history ---
with open(f'{results_dir}/fedavg_no_dp_history{suffix}.pkl', 'rb') as f:
    fedavg_history = pickle.load(f)
print(f"FedAvg history loaded: {type(fedavg_history).__name__}")

# --- Load DP-FedAvg histories (by file discovery, robust to filename differences) ---
dp_histories = {}
dp_files = sorted(glob.glob(f'{results_dir}/dp_*_history{suffix}.pkl'))
for fpath in dp_files:
    base = os.path.basename(fpath)
    if 'High' in base:
        label = 'High Privacy (ε ≈ 1)'
    elif 'Medium' in base:
        label = 'Medium Privacy (ε ≈ 3)'
    elif 'Low' in base:
        label = 'Low Privacy (ε ≈ 8)'
    else:
        label = base.replace('dp_', '').replace('_history' + suffix + '.pkl', '')
    with open(fpath, 'rb') as f:
        dp_histories[label] = pickle.load(f)
print(f"DP histories loaded: {list(dp_histories.keys())}")

In [ ]:
# Cell 2: Graph 1 - Training Loss Convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Centralized (need to track during training - placeholder)
ax = axes[0]
if hasattr(fedavg_history, 'losses_distributed') and fedavg_history.losses_distributed:
    rounds = [r for r, _ in fedavg_history.losses_distributed]
    losses = [l for _, l in fedavg_history.losses_distributed]
    ax.plot(rounds, losses, 'o-', label='FedAvg', linewidth=2, markersize=6)

for label, hist in dp_histories.items():
    if hist.losses_distributed:
        rounds = [r for r, _ in hist.losses_distributed]
        losses = [l for _, l in hist.losses_distributed]
        ax.plot(rounds, losses, 'o--', label=f'DP-FedAvg ({label})', linewidth=1.5, markersize=5)

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('Training Loss', fontsize=12)
ax.set_title('Graph 1: Training Loss Convergence', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Centralized baseline (horizontal line)
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Centralized (est.)')

# Per-round accuracy
ax = axes[1]
if fedavg_history.metrics_distributed.get('accuracy'):
    rounds = [r for r, _ in fedavg_history.metrics_distributed['accuracy']]
    accs = [a for _, a in fedavg_history.metrics_distributed['accuracy']]
    ax.plot(rounds, accs, 'o-', label='FedAvg', linewidth=2, markersize=6)

for label, hist in dp_histories.items():
    if hist.metrics_distributed.get('accuracy'):
        rounds = [r for r, _ in hist.metrics_distributed['accuracy']]
        accs = [a for _, a in hist.metrics_distributed['accuracy']]
        ax.plot(rounds, accs, 'o--', label=f'DP-FedAvg ({label})', linewidth=1.5, markersize=5)

ax.set_xlabel('Communication Round', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Graph 2: Per-Round Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{results_dir}/figures/graph1_2_convergence.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 3: Graph 3 - Privacy Budget vs F1 Trade-off
fig, ax = plt.subplots(figsize=(8, 6))

# Collect data
methods = ['Centralized', 'FedAvg', 'DP-FedAvg (ε≈8)', 'DP-FedAvg (ε≈3)', 'DP-FedAvg (ε≈1)']
epsilons = [float('inf'), float('inf'), 8, 3, 1]
f1_scores = []  # Fill from actual results

# Centralized
f1_scores.append(cent_results['centralized']['best_f1'])

# FedAvg
if fedavg_history.metrics_distributed.get('f1'):
    f1_scores.append(fedavg_history.metrics_distributed['f1'][-1][1])
else:
    f1_scores.append(0)

# DP variants
for label in ['Low Privacy (ε ≈ 8)', 'Medium Privacy (ε ≈ 3)', 'High Privacy (ε ≈ 1)']:
    if label in dp_histories:
        hist = dp_histories[label]
        if hist.metrics_distributed.get('f1'):
            f1_scores.append(hist.metrics_distributed['f1'][-1][1])
        else:
            f1_scores.append(0)
    else:
        f1_scores.append(0)

# Plot
colors = ['green', 'blue', 'orange', 'red', 'darkred']
markers = ['*', 'o', 's', '^', 'D']

for i, (method, eps, f1, color, marker) in enumerate(zip(methods, epsilons, f1_scores, colors, markers)):
    if eps == float('inf'):
        x_pos = 10
        label = f'{method} (no DP)'
    else:
        x_pos = eps
        label = method
    ax.scatter(x_pos, f1, s=200, c=color, marker=marker, label=label, zorder=5, edgecolors='white', linewidth=1.5)

# Connect DP points
dp_x = [e for e in epsilons if e != float('inf')]
dp_y = [f for e, f in zip(epsilons, f1_scores) if e != float('inf')]
if len(dp_x) > 1:
    ax.plot(dp_x, dp_y, 'k--', alpha=0.5, linewidth=1)

ax.set_xscale('log')
ax.set_xlabel('Privacy Budget (ε)', fontsize=13)
ax.set_ylabel('F1 Score', fontsize=13)
ax.set_title('Graph 3: Privacy-Utility Trade-off (ε vs F1)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)

# Add annotations
for i, (eps, f1) in enumerate(zip(epsilons, f1_scores)):
    if eps != float('inf'):
        ax.annotate(f'{f1:.3f}', (eps, f1), textcoords="offset points", xytext=(0,10), ha='center', fontsize=10)
    else:
        ax.annotate(f'{f1:.3f}', (10, f1), textcoords="offset points", xytext=(0,10), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{results_dir}/figures/graph3_privacy_utility.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 4: Graph 4 - Per-Client Accuracy (Non-IID Effect)
fig, ax = plt.subplots(figsize=(10, 6))

# This requires per-client evaluation - run a separate eval if needed
# For now, show the data distribution

client_labels = ['Hospital A\n(Disease-heavy)', 'Hospital B\n(Medicine-heavy)', 'Hospital C\n(Organ-heavy)']

# Placeholder - replace with actual per-client eval results
fedavg_client_acc = [0.78, 0.72, 0.68]  # Example values
dp_client_acc = [0.74, 0.68, 0.62]

x = np.arange(len(client_labels))
width = 0.35

bars1 = ax.bar(x - width/2, fedavg_client_acc, width, label='FedAvg (No DP)', color='steelblue', edgecolor='white')
bars2 = ax.bar(x + width/2, dp_client_acc, width, label='DP-FedAvg (ε≈3)', color='coral', edgecolor='white')

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Graph 4: Per-Client Performance (Non-IID Effect)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(client_labels, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(f'{results_dir}/figures/graph4_per_client.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 5: Graph 5 - Per-Entity-Type F1 Comparison
# Need to run detailed evaluation on best models
# This is a template - replace with actual per-entity results

entity_types = ['DISEASE', 'MEDICINE', 'ORGAN', 'SYMPTOM',
                'PHARMACOLOGICAL_CLASS', 'HORMONE', 'COMMON_MEDICAL_TERMS']

# Example data - REPLACE with actual results from model evaluation
centralized_f1 = [0.82, 0.79, 0.75, 0.77, 0.70, 0.65, 0.72]
fedavg_f1 = [0.78, 0.74, 0.70, 0.73, 0.66, 0.60, 0.68]
dp_f1 = [0.73, 0.69, 0.65, 0.68, 0.60, 0.55, 0.63]

x = np.arange(len(entity_types))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, centralized_f1, width, label='Centralized', color='green', alpha=0.8)
bars2 = ax.bar(x, fedavg_f1, width, label='FedAvg', color='blue', alpha=0.8)
bars3 = ax.bar(x + width, dp_f1, width, label='DP-FedAvg (ε≈3)', color='red', alpha=0.8)

ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('Graph 5: Per-Entity-Type F1 Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(entity_types, rotation=45, ha='right', fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.0)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{results_dir}/figures/graph5_per_entity.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6: Graph 6 - Summary Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Centralized\nBanglaBERT', 'FedAvg\n(3 clients,\nno DP)', 'DP-FedAvg\n(ε≈8)', 'DP-FedAvg\n(ε≈3)', 'DP-FedAvg\n(ε≈1)']
f1_vals = [f1_scores[0], f1_scores[1], f1_scores[2], f1_scores[3], f1_scores[4]]
colors = ['green', 'blue', 'orange', 'red', 'darkred']

bars = ax.bar(methods, f1_vals, color=colors, alpha=0.8, edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, f1_vals):
    ax.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, val),
                xytext=(0, 5), textcoords='offset points', ha='center', va='bottom',
                fontsize=13, fontweight='bold')

ax.set_ylabel('Macro F1 Score', fontsize=13)
ax.set_title('Graph 6: Overall Method Comparison', fontsize=15, fontweight='bold')
ax.set_ylim(0, max(f1_vals) * 1.2)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{results_dir}/figures/graph6_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 7: Generate Results Table for Thesis
table_data = {
    'Method': ['Centralized BanglaBERT', 'FedAvg (3 clients, no DP)',
               'DP-FedAvg (ε ≈ 1)', 'DP-FedAvg (ε ≈ 3)', 'DP-FedAvg (ε ≈ 8)'],
    'Accuracy': [f'{f1_scores[0]:.4f}', f'{f1_scores[1]:.4f}',
                 f'{f1_scores[2]:.4f}', f'{f1_scores[3]:.4f}', f'{f1_scores[4]:.4f}'],
    'Precision': ['-', '-', '-', '-', '-'],
    'Recall': ['-', '-', '-', '-', '-'],
    'F1': [f'{f1_scores[0]:.4f}', f'{f1_scores[1]:.4f}',
           f'{f1_scores[2]:.4f}', f'{f1_scores[3]:.4f}', f'{f1_scores[4]:.4f}'],
    'ε (Privacy)': ['∞ (no privacy)', '∞ (no formal DP)',
                    '~1.0', '~3.0', '~8.0']
}

df = pd.DataFrame(table_data)
print(df.to_markdown(index=False))

df.to_csv(f'{results_dir}/thesis_results_table.csv', index=False)
df.to_latex(f'{results_dir}/thesis_results_table.tex', index=False)

print("\nTable saved to thesis_results_table.csv and .tex")